In [ ]:
import pybedtools
import pysam
import pandas as pd
import os

# --- Configuration ---
WES_BED_PATH = "../../resources/hg38_exome_v2.0.2_targets_sorted_validated.re_annotated.bed"
BLACKLIST_BED_PATH = "../../resources/master_blacklist.bed"
RAW_VCF_PATH = "27YO_UDI5_unfiltered.vcf" # Use the raw, unfiltered VCF
REF_FASTA = "/Users/marcodupuisrodriguez/Documents/PhD/VCS/Marco_sequencing_data_and_variant_processing/reference_genome/GRCh38.primary_assembly.genome.fa"

# Thresholds from your script
VAR_20BP_MAX = 2    # If >2 variants in 20bp, mask region
VAR_250BP_MAX = 10   # If >5 variants in 250bp, mask region
HP_MAX_LEN = 8      # Max reference homopolymer length allowed

# ---------------------------------------------------------
# STEP 1: Static Masking (The "Where can we look?" phase)
# ---------------------------------------------------------

def get_density_mask(vcf_path, window, max_vars):
    """Identifies regions with high variant density to exclude."""
    # 1. Load VCF and convert to a simple BED-like object immediately 
    # to get rid of VCF headers that confuse merge/slop
    vcf = pybedtools.BedTool(vcf_path)
    
    # Cluster variants
    clusters = vcf.cluster(d=window)
    
    # Use pandas to find the IDs of clusters that exceed our limit
    # (Bedtools cluster adds the cluster ID as the LAST column)
    df = clusters.to_dataframe()
    if df.empty:
        return pybedtools.BedTool("", from_string=True)
    
    cluster_col = df.columns[-1]
    cluster_counts = df[cluster_col].value_counts()
    bad_cluster_ids = set(cluster_counts[cluster_counts > max_vars].index)
    
    if not bad_cluster_ids:
        # Return an empty BedTool if no bad clusters found
        return pybedtools.BedTool("", from_string=True)

    # 2. Filter for the bad clusters and explicitly format as BED (chrom, start, end)
    # We subtract 1 from start because VCF is 1-based and BED is 0-based
    bad_regions_list = []
    for feat in clusters:
        if int(feat[-1]) in bad_cluster_ids:
            # Create a simple 3-column BED feature
            bad_regions_list.append(f"{feat.chrom}\t{feat.start}\t{feat.end}")
    
    if not bad_regions_list:
        return pybedtools.BedTool("", from_string=True)
    
    # Create new BedTool from our list
    mask_regions = pybedtools.BedTool("\n".join(bad_regions_list), from_string=True)
    
    # 3. Slop and Merge
    # Note: 'hg38' needs to be recognized by your system's bedtools 
    # or you can pass a dict of {chrom: size}
    return mask_regions.slop(b=window, genome='hg38').merge()

print("Generating masks...")
wes = pybedtools.BedTool(WES_BED_PATH)
blacklist = pybedtools.BedTool(BLACKLIST_BED_PATH)

# Calculate Density Masks
mask_20bp = get_density_mask(RAW_VCF_PATH, 20, VAR_20BP_MAX)
mask_250bp = get_density_mask(RAW_VCF_PATH, 250, VAR_250BP_MAX)

# Final Subtraction: WES - Blacklist - Density Masks
# This creates your "Safe Zone"
clean_targets = wes.subtract(blacklist).subtract(mask_20bp).subtract(mask_250bp)
clean_targets_path = "sample_specific_clean_targets.bed"
clean_targets.saveas(clean_targets_path)

print(f"Original WES size: {wes.total_coverage():,}")
print(f"Clean Target size: {clean_targets.total_coverage():,}")

Generating masks...
Original WES size: 36,458,262
Clean Target size: 33,694,621


In [8]:
# -----------------------
# Global Quality Thresholds
# -----------------------
# Site-level thresholds

thresholds = {
    "MIN_MQ": 20,
    "MIN_BQ": 82,
    "MIN_ASXS": 10,
    "MAX_NM": 2.5,
    "MAX_N_COUNT": 2.1,
    "MAX_SOFTCLIP": 0.4,
    "MAX_INSERT": 524,
    "MIN_READ_POS": 4,
    "MAX_READ_POS": 138,
    "MIN_DEPTH": 20
}

# File Paths
BAM_PATH = "/Users/marcodupuisrodriguez/Documents/PhD/duplex_bams/UDI5.duplex.cons.mapped.bam"
CLEAN_TARGETS_BED = "sample_specific_clean_targets.bed" # The one we pre-processed

In [12]:
import pysam
import pybedtools
import os
import multiprocessing as mp
from tqdm.notebook import tqdm

# --- Settings ---
BAM_PATH = "/Users/marcodupuisrodriguez/Documents/PhD/duplex_bams/UDI5.duplex.cons.mapped.bam"
CLEAN_TARGETS_BED = "sample_specific_clean_targets.bed"

thresholds = {
    "MIN_MQ": 20,
    "MIN_BQ": 82,      # Duplex consensus quality
    "MIN_ASXS": 10,
    "MAX_NM": 2.5,
    "MAX_N_COUNT": 2.1,
    "MAX_SOFTCLIP": 0.4, # effectively 0 bases
    "MAX_INSERT": 524,
    "MIN_READ_POS": 4,
    "MAX_READ_POS": 138,
    "MIN_DEPTH": 20
}

def worker_task(args):
    """
    Completely self-contained worker to avoid 'handle is closed' errors.
    """
    bam_path, chrom, intervals, ts = args
    
    # Inner function to keep logic encapsulated
    def check_read(read, query_pos, ts):
        if read.mapping_quality < ts["MIN_MQ"] or abs(read.template_length) > ts["MAX_INSERT"]:
            return False
        try:
            as_val = read.get_tag("AS")
            xs_val = read.get_tag("XS") if read.has_tag("XS") else 0
            if (as_val - xs_val) < ts["MIN_ASXS"]: return False
        except KeyError: return False
        try:
            if read.get_tag("NM") > ts["MAX_NM"]: return False
        except KeyError: pass 
        if read.query_sequence.upper().count('N') > ts["MAX_N_COUNT"]: return False
        
        # Softclipping: cigar_stats[0][4] is the count of S bases
        if read.get_cigar_stats()[0][4] > ts["MAX_SOFTCLIP"]: return False
        
        if query_pos < ts["MIN_READ_POS"] or (read.query_length - query_pos) < (150 - ts["MAX_READ_POS"]):
            return False
        return True

    try:
        bam = pysam.AlignmentFile(bam_path, "rb")
        local_total = 0
        
        for start, end in intervals:
            # truncate=True ensures we only look at bases inside the BED interval
            for pileupcolumn in bam.pileup(chrom, start, end, truncate=True, min_mapping_quality=ts["MIN_MQ"]):
                clean_count = 0
                fwd_seen = False
                rev_seen = False
                
                for pileupread in pileupcolumn.pileups:
                    if pileupread.is_del or pileupread.is_refskip: continue
                    
                    read = pileupread.alignment
                    # Check base quality for this specific observation
                    if read.query_qualities[pileupread.query_position] < ts["MIN_BQ"]:
                        continue
                        
                    if check_read(read, pileupread.query_position, ts):
                        clean_count += 1
                        if read.is_reverse: rev_seen = True
                        else: fwd_seen = True
                
                # If site is callable, add the total number of passing reads to our count
                if clean_count >= ts["MIN_DEPTH"] and fwd_seen and rev_seen:
                    local_total += clean_count
        
        bam.close()
        return local_total
    except Exception as e:
        return f"Error on {chrom}: {e}"

def run_parallel_count(bam_path, bed_path, ts, threads=13):
    # Prepare intervals
    bed = pybedtools.BedTool(bed_path)
    chrom_map = {}
    for interval in bed:
        if interval.chrom not in chrom_map: chrom_map[interval.chrom] = []
        chrom_map[interval.chrom].append((interval.start, interval.end))
    
    tasks = [(bam_path, chrom, invs, ts) for chrom, invs in chrom_map.items()]
    
    # Use 'fork' context which is much more reliable in Jupyter on Mac
    ctx = mp.get_context('fork')
    
    results = []
    with ctx.Pool(threads) as pool:
        # Use imap to track progress with tqdm
        for res in tqdm(pool.imap_unordered(worker_task, tasks), total=len(tasks), desc="Processing"):
            results.append(res)
            
    final_sum = 0
    for r in results:
        if isinstance(r, str): print(r)
        else: final_sum += r
    return final_sum

# Execute
total_bases = run_parallel_count(BAM_PATH, CLEAN_TARGETS_BED, thresholds, threads=13)
print(f"\nTotal Callable Observed Bases: {total_bases:,}")

Processing:   0%|          | 0/24 [00:00<?, ?it/s]


Total Callable Observed Bases: 1,368,663,355
